# Slate Forensics — reference solution

**Task**: for every logged recommendation request, decide whether it was served
by the healthy policy or a broken variant, name the failure mode, and
reconstruct the healthy slate.

**Approach** (three stages):
1. **Policy imitation** — learn to rank catalog items the way the healthy
   service did, from the labeled healthy slates: multi-window co-purchase
   affinity, popularity bands, price-band matching and ownership features feed
   a gradient-boosted ranker (customer-disjoint validation).
2. **Forensic diagnosis** — slate-level features compare each emitted slate
   against the imitated policy and against known failure signatures
   (popularity concentration, price-band displacement, seasonal staleness);
   a 4-class model outputs `none` / `popularity_fallback` / `price_band_shift`
   / `stale_index`.
3. **Consistency-aware assembly** — clean requests resubmit the emitted slate;
   corrupted requests submit the imitated healthy slate; the corruption
   threshold is tuned on held-out customers by scoring full pseudo-submissions
   with the official `grade.py`.

Reads only `./dataset/public/`, writes `./working/submission.csv`.
Only Kaggle-image libraries (pandas / numpy / scipy / lightgbm with an
sklearn fallback). Runs in well under 30 minutes.

In [1]:
# Slate Forensics — reference solution
# Strategy:
#   1. Imitate the healthy recommender: learn a candidate ranker from the
#      labeled healthy slates (policy imitation with multi-window
#      co-purchase, price and popularity features).
#   2. Forensic audit: compare each emitted slate against the imitated
#      policy and engineered failure signatures; a multiclass model assigns
#      none / popularity_fallback / price_band_shift / stale_index.
#   3. Consistency-aware assembly: clean -> resubmit emitted slate;
#      corrupted -> submit the imitated healthy slate.

import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

T0 = time.time()
SEED = 7
DATA = Path("./dataset/public")
WORK = Path("./working")
WORK.mkdir(exist_ok=True)

MODES = ["popularity_fallback", "price_band_shift", "stale_index"]
TOP_K = 10

tx = pd.read_csv(DATA / "transactions.csv", parse_dates=["invoice_date"],
                 dtype={"invoice": str, "stock_code": str,
                        "customer_id": str}, keep_default_na=False)
catalog = pd.read_csv(DATA / "catalog.csv", dtype={"stock_code": str})
slates_train = pd.read_csv(DATA / "slates_train.csv",
                           dtype={"slate_id": str, "customer_id": str,
                                  "stock_code": str})
labels_train = pd.read_csv(DATA / "slate_labels_train.csv",
                           dtype={"slate_id": str, "mode": str})
healthy_train = pd.read_csv(DATA / "healthy_slates_train.csv",
                            dtype={"slate_id": str, "stock_code": str})
slates_test = pd.read_csv(DATA / "slates_test.csv",
                          dtype={"slate_id": str, "customer_id": str,
                                 "stock_code": str})
print(f"tx={len(tx):,}  train slates={slates_train.slate_id.nunique():,}  "
      f"test slates={slates_test.slate_id.nunique():,}  "
      f"[{time.time()-T0:.0f}s]")

tx=993,283  train slates=5,244  test slates=757  [2s]


In [2]:
# Sales rows usable for behavioural statistics: positive quantity/price,
# non-cancelled invoices, standard 5-digit product codes.
sale = tx[(tx.quantity > 0) & (tx.unit_price > 0)
          & ~tx.invoice.str.startswith("C")
          & tx.stock_code.str.match(r"^\d{5}[A-Z]*$")].copy()

# Window bands (days back from the anchor) used for statistics. B0 is the
# freshest quarter; B1/B2 look further back (seasonality forensics); YEAR
# is the full lookback.
BANDS = {"b0": (0, 90), "b1": (90, 180), "b2": (180, 270), "yr": (0, 365)}


class BandStats:
    """Item supports and item-item cosine for one (anchor, band)."""

    def __init__(self, rows: pd.DataFrame, with_cosine: bool):
        pairs = rows[["invoice", "stock_code"]].drop_duplicates()
        self.items = np.array(sorted(pairs.stock_code.unique()))
        self.index = {c: i for i, c in enumerate(self.items)}
        inv, _ = pd.factorize(pairs.invoice, sort=True)
        col = pairs.stock_code.map(self.index).to_numpy()
        B = sparse.csr_matrix((np.ones(len(pairs), np.float32), (inv, col)),
                              shape=(inv.max() + 1 if len(inv) else 0,
                                     len(self.items)))
        self.support = np.asarray(B.sum(axis=0)).ravel()
        self.cosine = None
        if with_cosine and len(self.items):
            C = (B.T @ B).toarray().astype(np.float32)
            np.fill_diagonal(C, 0.0)
            d = np.sqrt(np.maximum(self.support, 1.0))
            self.cosine = C / d[None, :] / d[:, None]

    def sup_of(self, codes: np.ndarray) -> np.ndarray:
        idx = pd.Series(np.arange(len(self.items)), index=self.items)
        j = idx.reindex(codes).to_numpy()
        out = np.zeros(len(codes))
        ok = ~np.isnan(j)
        out[ok] = self.support[j[ok].astype(int)]
        return out


def anchor_stats(anchor: pd.Timestamp) -> dict:
    look = sale[(sale.invoice_date < anchor)
                & (sale.invoice_date >= anchor - pd.Timedelta(days=365))
                & (sale.customer_id != "")]
    bands = {}
    for name, (lo, hi) in BANDS.items():
        rows = look[(look.invoice_date < anchor - pd.Timedelta(days=lo))
                    & (look.invoice_date >= anchor - pd.Timedelta(days=hi))]
        bands[name] = BandStats(rows, with_cosine=name in ("b0", "yr"))
    item_price = look.groupby("stock_code").unit_price.median()
    cust_price = look.groupby("customer_id").unit_price.median()
    lastbuy = (look.groupby(["customer_id", "stock_code"]).invoice_date.max()
               .reset_index())
    lastbuy["age_d"] = (anchor - lastbuy.invoice_date).dt.total_seconds() / 86400
    return {"bands": bands, "item_price": item_price,
            "cust_price": cust_price,
            "hist": dict(tuple(lastbuy.groupby("customer_id")))}

In [3]:
# Candidate generation + item-level features for every slate. Candidates are
# the union of: top-N by history-affinity in the fresh band, top-N in the
# yearly band, the anchor's popularity head, and the emitted slate itself.
# Anchors are processed one at a time so only one anchor's dense cosine
# matrices live in memory.
N_AFF, N_POP = 150, 60


def hist_vector(st, hist_df, band):
    b = st["bands"][band]
    if b.cosine is None or hist_df is None:
        return None
    h = hist_df.sort_values(["age_d", "stock_code"]).head(40)
    w = 0.5 ** (h.age_d.to_numpy() / 60.0)
    cols = [b.index.get(c, -1) for c in h.stock_code]
    ok = np.array(cols) >= 0
    if not ok.any():
        return np.zeros(len(b.items))
    return b.cosine[:, np.array(cols)[ok]] @ w[ok].astype(np.float32)


def slate_frame(slates: pd.DataFrame) -> pd.DataFrame:
    meta = slates.drop_duplicates("slate_id")[
        ["slate_id", "customer_id", "anchor_date"]]
    emitted = (slates.sort_values(["slate_id", "position"])
               .groupby("slate_id").stock_code.agg(list))
    meta = meta.set_index("slate_id")
    meta["emitted"] = emitted
    return meta


meta_train, meta_test = slate_frame(slates_train), slate_frame(slates_test)
healthy_map = (healthy_train.sort_values(["slate_id", "position"])
               .groupby("slate_id").stock_code.agg(list))


def build_candidates(meta: pd.DataFrame, st: dict,
                     healthy: pd.Series | None):
    rows = []
    for sid, r in meta.iterrows():
        hist = st["hist"].get(r.customer_id)
        s0 = hist_vector(st, hist, "b0")
        syr = hist_vector(st, hist, "yr")
        b0, byr = st["bands"]["b0"], st["bands"]["yr"]
        cand = set(r.emitted)
        if s0 is not None:
            cand |= set(b0.items[np.argsort(-s0)[:N_AFF]])
        if syr is not None:
            cand |= set(byr.items[np.argsort(-syr)[:N_AFF]])
        cand |= set(b0.items[np.argsort(-b0.support)[:N_POP]])
        cand = sorted(cand)

        codes = np.array(cand)
        f = pd.DataFrame({"slate_id": sid, "stock_code": codes})
        idx0 = pd.Series(np.arange(len(b0.items)), index=b0.items)
        j0 = idx0.reindex(codes).to_numpy()
        ok0 = ~np.isnan(j0)
        aff0 = np.zeros(len(codes))
        if s0 is not None:
            aff0[ok0] = s0[j0[ok0].astype(int)]
        idxy = pd.Series(np.arange(len(byr.items)), index=byr.items)
        jy = idxy.reindex(codes).to_numpy()
        oky = ~np.isnan(jy)
        affy = np.zeros(len(codes))
        if syr is not None:
            affy[oky] = syr[jy[oky].astype(int)]
        f["aff_b0"], f["aff_yr"] = aff0, affy
        f["aff_b0_rank"] = (-aff0).argsort().argsort() / max(len(codes), 1)
        for name in BANDS:
            f[f"logsup_{name}"] = np.log1p(st["bands"][name].sup_of(codes))
        f["season_ratio"] = (f.logsup_b2 - f.logsup_b0)
        p_item = st["item_price"].reindex(codes).to_numpy()
        p_cust = st["cust_price"].get(r.customer_id, np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            lr = np.log(p_item / p_cust)
        f["price_lr"] = np.where(np.isfinite(lr), lr, 0.0)
        f["price_alr"] = np.abs(f.price_lr)
        owned = set(hist.stock_code) if hist is not None else set()
        recent = (set(hist[hist.age_d <= 90].stock_code)
                  if hist is not None else set())
        f["owned_yr"] = f.stock_code.isin(owned).astype(int)
        f["owned_90"] = f.stock_code.isin(recent).astype(int)
        if healthy is not None:
            f["y"] = f.stock_code.isin(set(healthy[sid])).astype(int)
        rows.append(f)
    return pd.concat(rows, ignore_index=True)


anchors = sorted(set(meta_train.anchor_date) | set(meta_test.anchor_date))
TOP_POP = {}                       # anchor -> (top10 set, top50 set)
ctr_parts, cte_parts = [], []
for a in anchors:
    st = anchor_stats(pd.Timestamp(a))
    b0 = st["bands"]["b0"]
    TOP_POP[a] = (set(b0.items[np.argsort(-b0.support)[:TOP_K]]),
                  set(b0.items[np.argsort(-b0.support)[:50]]))
    mtr = meta_train[meta_train.anchor_date == a]
    mte = meta_test[meta_test.anchor_date == a]
    if len(mtr):
        ctr_parts.append(build_candidates(mtr, st, healthy_map))
    if len(mte):
        cte_parts.append(build_candidates(mte, st, None))
    del st
    print(f"  anchor {a} done [{time.time()-T0:.0f}s]")
cand_train = pd.concat(ctr_parts, ignore_index=True)
cand_test = pd.concat(cte_parts, ignore_index=True)
del ctr_parts, cte_parts
cover = cand_train.groupby("slate_id").y.sum().mean()
print(f"candidates: train={len(cand_train):,} test={len(cand_test):,} | "
      f"healthy items inside candidate pool: {cover:.2f}/10 "
      f"[{time.time()-T0:.0f}s]")

  anchor 2010-09-01 done [5s]


  anchor 2010-12-01 done [10s]


  anchor 2011-03-01 done [15s]


  anchor 2011-06-01 done [19s]


  anchor 2011-09-01 done [23s]


  anchor 2011-11-15 done [31s]
candidates: train=1,085,034 test=161,519 | healthy items inside candidate pool: 9.98/10 [31s]


In [4]:
# Policy-imitation ranker. Customer-disjoint validation guards against
# memorising customers instead of learning the policy.
FEATS = ["aff_b0", "aff_yr", "aff_b0_rank", "logsup_b0", "logsup_b1",
         "logsup_b2", "logsup_yr", "season_ratio", "price_lr", "price_alr",
         "owned_yr", "owned_90"]

cust_of = meta_train.customer_id
val_cust = set(pd.Series(sorted(cust_of.unique()))
               .sample(frac=0.2, random_state=SEED))
val_ids = set(cust_of[cust_of.isin(val_cust)].index)
tr_mask = ~cand_train.slate_id.isin(val_ids)

try:
    import lightgbm as lgb
    ranker = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.06, num_leaves=63,
        min_child_samples=40, subsample=0.9, subsample_freq=1,
        colsample_bytree=0.9, random_state=SEED, deterministic=True,
        force_row_wise=True, n_jobs=4, verbose=-1)
except Exception:                                    # Kaggle image fallback
    from sklearn.ensemble import HistGradientBoostingClassifier
    ranker = HistGradientBoostingClassifier(max_iter=400, random_state=SEED)

ranker.fit(cand_train.loc[tr_mask, FEATS], cand_train.loc[tr_mask, "y"])
cand_train["p"] = ranker.predict_proba(cand_train[FEATS])[:, 1]
cand_test["p"] = ranker.predict_proba(cand_test[FEATS])[:, 1]


def predicted_slate(cands: pd.DataFrame) -> pd.Series:
    out = {}
    for sid, grp in cands.groupby("slate_id"):
        g = grp.sort_values(["p", "logsup_b0", "stock_code"],
                            ascending=[False, False, True])
        out[sid] = g.stock_code.head(TOP_K).tolist()
    return pd.Series(out)


pred_train, pred_test = predicted_slate(cand_train), predicted_slate(cand_test)


def rbo10(a, b, p=0.9):
    w = p ** np.arange(TOP_K)
    sa, sb, inter, acc = set(), set(), 0, 0.0
    for d in range(TOP_K):
        inter += (a[d] in sb) + (b[d] in sa) + (a[d] == b[d])
        sa.add(a[d]); sb.add(b[d])
        acc += w[d] * inter / (d + 1)
    return acc / w.sum()


val_rbo = np.mean([rbo10(pred_train[s], healthy_map[s])
                   for s in sorted(val_ids)])
print(f"held-out customers: imitation RBO vs healthy = {val_rbo:.3f} "
      f"[{time.time()-T0:.0f}s]")

held-out customers: imitation RBO vs healthy = 0.744 [47s]


In [5]:
# Slate-level forensic features + 4-class diagnosis model.
def slate_features(meta, cands, pred) -> pd.DataFrame:
    by_item = cands.set_index(["slate_id", "stock_code"])
    rows = []
    for sid, r in meta.iterrows():
        em = r.emitted
        sub = by_item.loc[sid].reindex(em)
        top_pop, top50 = TOP_POP[r.anchor_date]
        rows.append({
            "slate_id": sid,
            "rbo_em_pred": rbo10(em, pred[sid]),
            "jac_em_pred": len(set(em) & set(pred[sid])) / TOP_K,
            "aff_b0_mean": sub.aff_b0.mean(),
            "aff_yr_mean": sub.aff_yr.mean(),
            "price_lr_med": sub.price_lr.median(),
            "price_lr_mean": sub.price_lr.mean(),
            "price_hi_frac": (sub.price_lr > 0.6).mean(),
            "logsup_b0_mean": sub.logsup_b0.mean(),
            "season_mean": sub.season_ratio.mean(),
            "season_hi_frac": (sub.season_ratio > 0.5).mean(),
            "owned90_frac": sub.owned_90.mean(),
            "in_top10_frac": np.mean([c in top_pop for c in em]),
            "in_top50_frac": np.mean([c in top50 for c in em]),
        })
    return pd.DataFrame(rows).set_index("slate_id")


sf_train = slate_features(meta_train, cand_train, pred_train)
sf_test = slate_features(meta_test, cand_test, pred_test)
y_mode = labels_train.set_index("slate_id")["mode"].reindex(sf_train.index)
classes = ["none"] + MODES
y_enc = y_mode.map({c: i for i, c in enumerate(classes)})

tr2 = ~sf_train.index.isin(val_ids)
try:
    import lightgbm as lgb
    clf = lgb.LGBMClassifier(
        objective="multiclass", num_class=4, n_estimators=300,
        learning_rate=0.07, num_leaves=31, min_child_samples=25,
        random_state=SEED, deterministic=True, force_row_wise=True,
        n_jobs=4, verbose=-1)
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    clf = HistGradientBoostingClassifier(max_iter=300, random_state=SEED)

clf.fit(sf_train[tr2], y_enc[tr2])
proba_val = clf.predict_proba(sf_train[~tr2])
proba_test = clf.predict_proba(sf_test)
print(f"diagnosis model trained [{time.time()-T0:.0f}s]")

diagnosis model trained [54s]


In [6]:
# Metric-aware assembly. The corruption threshold is tuned on the held-out
# customers by scoring full pseudo-submissions with the official grader.
import importlib.util
spec = importlib.util.spec_from_file_location("grade", "./grade.py")
grade_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(grade_mod)


def assemble(meta, pred, proba, ids, thr) -> pd.DataFrame:
    p_none = proba[:, 0]
    mode_ix = 1 + np.argmax(proba[:, 1:], axis=1)
    rows = []
    for k, sid in enumerate(ids):
        corrupted = int(1 - p_none[k] > thr)
        mode = classes[mode_ix[k]] if corrupted else "none"
        slate = list(meta.loc[sid, "emitted"]) if not corrupted \
            else list(pred[sid])
        if corrupted and slate == list(meta.loc[sid, "emitted"]):
            alt = [c for c in pred[sid] if c not in slate]
            slate = slate[:-1] + (alt[:1] or ["99999X"])
        for pos, code in enumerate(slate, start=1):
            rows.append({"slate_id": sid, "corrupted": corrupted,
                         "mode": mode, "position": pos, "stock_code": code})
    return pd.DataFrame(rows)


val_sorted = sorted(val_ids)
em_val = meta_train.loc[val_sorted, "emitted"]
lbl_ix = labels_train.set_index("slate_id")
pseudo_ans = []
for sid in val_sorted:
    lbl = lbl_ix.loc[sid]
    for pos in range(TOP_K):
        pseudo_ans.append({
            "slate_id": sid, "corrupted": lbl.corrupted, "mode": lbl["mode"],
            "position": pos + 1, "healthy_code": healthy_map[sid][pos],
            "emitted_code": em_val[sid][pos]})
pseudo_ans = pd.DataFrame(pseudo_ans)

best_thr, best_s = 0.5, -1
for thr in np.round(np.arange(0.30, 0.71, 0.05), 2):
    sub = assemble(meta_train, pred_train, proba_val, val_sorted, thr)
    s = grade_mod.grade(sub, pseudo_ans)
    if s > best_s:
        best_thr, best_s = thr, s
print(f"validation composite={best_s:.4f} at threshold={best_thr} "
      f"[{time.time()-T0:.0f}s]")

validation composite=0.8936 at threshold=0.7 [55s]


In [7]:
test_ids = sorted(sf_test.index)
submission = assemble(meta_test, pred_test, proba_test, test_ids, best_thr)
submission.to_csv(WORK / "submission.csv", index=False)
print(f"wrote {WORK/'submission.csv'} rows={len(submission)} "
      f"[{time.time()-T0:.0f}s]")

wrote working/submission.csv rows=7570 [55s]
